<a href="https://colab.research.google.com/github/waltergsotelo/aprendizaje_profundo/blob/main/TP_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import os
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Verificar si hay GPU disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


## Hacemos la descarga los de los archivo .wav separados por carpeta segun el genero

In [33]:
import gdown
import zipfile
import os

url = "https://drive.google.com/uc?id=1txyVxGVezQ4fcHqFi-n3gcWKEYQexu1_"
output = "dataset_music.zip"

gdown.download(url, output, quiet=False)

destino = "datos_music"
os.makedirs(destino, exist_ok=True)

with zipfile.ZipFile(output, "r") as zip_ref:
    zip_ref.extractall(destino)

DATASET_PATH = "/content/datos_music/genres_original"

Downloading...
From (original): https://drive.google.com/uc?id=1txyVxGVezQ4fcHqFi-n3gcWKEYQexu1_
From (redirected): https://drive.google.com/uc?id=1txyVxGVezQ4fcHqFi-n3gcWKEYQexu1_&confirm=t&uuid=46c0930a-0562-4228-8eba-caebc3fc1852
To: /content/dataset_music.zip
100%|██████████| 1.22G/1.22G [00:20<00:00, 58.4MB/s]


### Vamos a separa los archivos de cada estilo musical en un 80/20 para training/test

In [34]:
import os
import glob
from sklearn.model_selection import train_test_split

# 1. Ruta a tu carpeta principal que contiene las subcarpetas con géneros
data_dir = DATASET_PATH # Cambia esto por tu ruta real

audio_paths = []
labels = []
class_names = sorted(os.listdir(data_dir))
class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}

print(f"Géneros detectados: {class_names}")

# 2. Recorrer las subcarpetas y recolectar todas las rutas de archivos .wav y sus etiquetas
for genre in class_names:
    genre_path = os.path.join(data_dir, genre)
    if os.path.isdir(genre_path):
        # Buscar todos los archivos .wav en la carpeta del género
        wav_files = glob.glob(os.path.join(genre_path, "*.wav"))
        for file in wav_files:
            audio_paths.append(file)
            labels.append(class_to_idx[genre])

print(f"Total de archivos encontrados: {len(audio_paths)}")

# 3. Hacer la partición 80% entrenamiento / 20% validación (con estratificación para mantener proporciones)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    audio_paths,
    labels,
    test_size=0.20,      # 20% para validación
    random_state=42,     # Semilla para que la división sea reproducible
    stratify=labels      # Asegura que cada género tenga el mismo % en train y val
)

print(f"Archivos de Entrenamiento: {len(train_paths)}")
print(f"Archivos de Validación: {len(val_paths)}")

Géneros detectados: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Total de archivos encontrados: 999
Archivos de Entrenamiento: 799
Archivos de Validación: 200


In [35]:
from tqdm import tqdm
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset

class MusicGenreDataset(Dataset):
    def __init__(self, audio_paths, labels, sr=22050, duration=30, n_mels=128, max_time_frames=1300):
        self.audio_paths = audio_paths
        self.labels = labels
        self.sr = sr
        self.duration = duration
        self.n_mels = n_mels
        self.max_time_frames = max_time_frames

        # Inicializar listas para almacenar en memoria RAM
        self.cached_specs = []
        self.cached_labels = []

        print(f"⏳ Pre-calculando y cargando {len(audio_paths)} audios en memoria RAM...")
        for idx in tqdm(range(len(self.audio_paths))):
            audio_path = self.audio_paths[idx]
            label = self.labels[idx]

            try:
                # Cargar el archivo de audio (.wav)
                y, sr_loaded = librosa.load(audio_path, sr=self.sr, duration=self.duration)

                # Extraer el espectrograma de Mel
                mel_spec = librosa.feature.melspectrogram(y=y, sr=sr_loaded, n_fft=2048, hop_length=512, n_mels=self.n_mels)
                mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

                # Normalización estándar (Z-score)
                mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-6)

                # Ajustar longitud temporal (Padding / Truncate)
                current_frames = mel_spec_db.shape[1]
                if current_frames < self.max_time_frames:
                    pad_width = self.max_time_frames - current_frames
                    mel_spec_db = np.pad(mel_spec_db, ((0, 0), (0, pad_width)), mode='constant')
                else:
                    mel_spec_db = mel_spec_db[:, :self.max_time_frames]

                # Convertir a Tensores de PyTorch
                tensor_spec = torch.tensor(mel_spec_db, dtype=torch.float32).unsqueeze(0) # [1, n_mels, time_frames]
                label_tensor = torch.tensor(label, dtype=torch.long)

            except Exception as e:
                # En caso de un error de lectura, usar un tensor de ceros para evitar detener el entrenamiento
                tensor_spec = torch.zeros((1, self.n_mels, self.max_time_frames), dtype=torch.float32)
                label_tensor = torch.tensor(0, dtype=torch.long)

            self.cached_specs.append(tensor_spec)
            self.cached_labels.append(label_tensor)

        print("✅ ¡Todos los espectrogramas han sido pre-calculados con éxito en RAM!")

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, idx):
        # Retorno inmediato ultra-eficiente desde la memoria
        return self.cached_specs[idx], self.cached_labels[idx]

In [36]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(TemporalAttention, self).__init__()
        self.attention_weights = nn.Linear(hidden_dim, 1)

    def forward(self, rnn_out):
        # rnn_out shape: [Batch, Time_Frames, Hidden_Dim]
        scores = self.attention_weights(rnn_out) # [Batch, Time_Frames, 1]
        weights = F.softmax(scores, dim=1)        # [Batch, Time_Frames, 1]

        # Suma ponderada temporal (Context Vector)
        context_vector = torch.sum(rnn_out * weights, dim=1) # [Batch, Hidden_Dim]
        return context_vector, weights

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Conexión residual si el número de canales cambia
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out

class ImprovedMusicGenreCRNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super(ImprovedMusicGenreCRNN, self).__init__()

        # Extractor Convolucional Mejorado con Bloques Residuales y Dropout
        self.layer1 = ConvBlock(1, 32)
        self.pool1 = nn.MaxPool2d((2, 2))  # 128 -> 64
        self.drop1 = nn.Dropout2d(dropout)

        self.layer2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d((2, 2))  # 64 -> 32
        self.drop2 = nn.Dropout2d(dropout)

        self.layer3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d((2, 2))  # 32 -> 16
        self.drop3 = nn.Dropout2d(dropout)

        self.layer4 = ConvBlock(128, 256)
        self.pool4 = nn.MaxPool2d((2, 2))  # 16 -> 8
        self.drop4 = nn.Dropout2d(dropout)

        # Entrada a la RNN: 256 canales * 8 frecuencias
        self.rnn_input_size = 256 * 8

        # Red Recurrente bidireccional LSTM de 2 capas para modelar la temporalidad del ritmo
        self.rnn = nn.LSTM(
            input_size=self.rnn_input_size,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        # Mecanismo de atención temporal
        self.attention = TemporalAttention(hidden_dim=256)

        # Clasificador final robusto con Dropout
        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        # x shape: [Batch, 1, 128, Time_Frames]
        batch_size = x.size(0)

        # Feature Extraction con CNN
        x = self.drop1(self.pool1(self.layer1(x)))
        x = self.drop2(self.pool2(self.layer2(x)))
        x = self.drop3(self.pool3(self.layer3(x)))
        x = self.drop4(self.pool4(self.layer4(x)))

        # Reordenar dimensiones para la RNN
        x = x.permute(0, 3, 1, 2) # [Batch, Reduced_Time, 256, 8]
        x = x.reshape(batch_size, x.size(1), -1) # [Batch, Reduced_Time, 256 * 8]

        # Procesamiento temporal con LSTM
        rnn_out, _ = self.rnn(x) # [Batch, Reduced_Time, 256]

        # Capa de Atención temporal
        context_vector, attention_weights = self.attention(rnn_out)

        # Clasificación
        out = self.fc(context_vector)
        return out, attention_weights

In [37]:
# --- CONFIGURACIÓN REAL CON TUS ARCHIVOS .WAV ---

# 1. Crear los Datasets usando los splits 80/20 que hicimos antes
train_dataset = MusicGenreDataset(audio_paths=train_paths, labels=train_labels)
val_dataset = MusicGenreDataset(audio_paths=val_paths, labels=val_labels)

# 2. Crear los DataLoaders (shufle=True solo para entrenamiento)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# 3. Instanciar modelo, función de pérdida y optimizador
num_genres = len(class_names)  # Se adapta automáticamente a cuántas carpetas/géneros tengas
model = ImprovedMusicGenreCRNN(num_classes=num_genres, dropout=0.3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"¡Modelo inicializado con éxito para {num_genres} géneros: {class_names}!")
print(f"Lotes de entrenamiento: {len(train_loader)} | Lotes de validación: {len(val_loader)}")

⏳ Pre-calculando y cargando 799 audios en memoria RAM...


100%|██████████| 799/799 [00:43<00:00, 18.40it/s]


✅ ¡Todos los espectrogramas han sido pre-calculados con éxito en RAM!
⏳ Pre-calculando y cargando 200 audios en memoria RAM...


100%|██████████| 200/200 [00:10<00:00, 18.26it/s]

✅ ¡Todos los espectrogramas han sido pre-calculados con éxito en RAM!
¡Modelo inicializado con éxito para 10 géneros: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']!
Lotes de entrenamiento: 100 | Lotes de validación: 25


In [38]:
# --- INICIALIZACIÓN DEL MODELO ACTIVO (ImprovedMusicGenreCRNN) ---

# Instanciamos la nueva arquitectura unificada con bloques residuales y dropout
model = ImprovedMusicGenreCRNN(num_classes=num_genres, dropout=0.3).to(device)

# Definimos la función de pérdida y el optimizador para esta nueva arquitectura
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("✅ ¡Arquitectura residual 'ImprovedMusicGenreCRNN' instanciada y lista para entrenar!")

✅ ¡Arquitectura residual 'ImprovedMusicGenreCRNN' instanciada y lista para entrenar!


In [40]:
# --- OPTIMIZACIÓN DEL PROCESO DE ENTRENAMIENTO PARA LA RED MEJORADA ---

# Usamos AdamW (Adam con Weight Decay) para regularizar los pesos del extractor residual
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

# Scheduler dinámico: Reduce el LR por un factor de 0.5 si la pérdida de validación no mejora en 3 épocas
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

print("⚙️ Optimizador AdamW y Scheduler ReduceLROnPlateau configurados con éxito.")

⚙️ Optimizador AdamW y Scheduler ReduceLROnPlateau configurados con éxito.


In [ ]:
import time

# Ampliamos a 40 épocas para dar tiempo a la convergencia de las conexiones residuales
num_epochs = 40
best_val_accuracy = 0.0

# Historial para graficar
train_loss_history = []
val_loss_history = []
train_acc_history = []
val_acc_history = []

print(f"🚀 Iniciando entrenamiento optimizado por {num_epochs} épocas con Scheduler...")

for epoch in range(num_epochs):
    start_time = time.time()

    # ==========================
    # 1. FASE DE ENTRENAMIENTO
    # ==========================
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs, _ = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        # Gradient clipping para estabilizar el entrenamiento de la LSTM bidireccional
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100 * correct_train / total_train

    # ==========================
    # 2. FASE DE VALIDACIÓN
    # ==========================
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / total_val
    epoch_val_acc = 100 * correct_val / total_val

    # Guardar en el historial
    train_loss_history.append(epoch_train_loss)
    val_loss_history.append(epoch_val_loss)
    train_acc_history.append(epoch_train_acc)
    val_acc_history.append(epoch_val_acc)

    # Actualizar el learning rate basado en la pérdida de validación
    scheduler.step(epoch_val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    elapsed_time = time.time() - start_time

    print(f"Época [{epoch+1}/{num_epochs}] ({elapsed_time:.1f}s) | LR: {current_lr:.6f} | ")

🚀 Iniciando entrenamiento optimizado por 40 épocas con Scheduler...
Época [1/40] (26.7s) | LR: 0.001000 | 
Época [2/40] (26.3s) | LR: 0.001000 | 
Época [3/40] (27.0s) | LR: 0.001000 | 
Época [4/40] (27.3s) | LR: 0.001000 | 


In [ ]:
import time

# --- CONFIGURACIÓN DEL ENTRENAMIENTO ---
num_epochs = 20          # Puedes ajustar el número de épocas según tu dataset
best_val_accuracy = 0.0

print(f"🚀 Iniciando entrenamiento por {num_epochs} épocas...\n")

for epoch in range(num_epochs):
    start_time = time.time()

    # ==========================
    # 1. FASE DE ENTRENAMIENTO
    # ==========================
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Limpiar gradientes anteriores
        optimizer.zero_grad()

        # Forward pass (el modelo devuelve outputs y attention_weights)
        outputs, _ = model(inputs)

        # Calcular pérdida
        loss = criterion(outputs, labels)

        # Backward pass y optimización
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

        # Calcular aciertos en train
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100 * correct_train / total_train

    # ==========================
    # 2. FASE DE VALIDACIÓN
    # ==========================
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs, _ = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    epoch_val_loss = val_loss / total_val
    epoch_val_acc = 100 * correct_val / total_val

    elapsed_time = time.time() - start_time

    # Imprimir métricas de la época
    print(f"Época [{epoch+1}/{num_epochs}] ({elapsed_time:.1f}s) | "
          f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.2f}%")

    # ==========================
    # 3. GUARDAR EL MEJOR MODELO
    # ==========================
    if epoch_val_acc > best_val_accuracy:
        best_val_accuracy = epoch_val_acc
        torch.save(model.state_dict(), "mejor_modelo_crnn.pth")
        print(f"   ⭐ ¡Nuevo mejor modelo guardado! (Val Acc: {epoch_val_acc:.2f}%)")

print(f"\n✨ ¡Entrenamiento finalizado! Mejor precisión en validación: {best_val_accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Valores obtenidos del último entrenamiento registrado en el historial:
train_losses = [1.7952, 1.4210, 1.2114, 1.0823, 0.9963, 0.8731, 0.7993, 0.6554, 0.6553, 0.5589, 0.5678, 0.5357, 0.4139, 0.3635, 0.2949]
val_losses = [1.6339, 1.3642, 1.3001, 1.0741, 1.0023, 1.0562, 0.8112, 0.7770, 0.8427, 0.9101, 0.9762, 0.7842, 0.8757, 0.7258, 0.8324]
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses, 'b-o', label='Pérdida Entrenamiento (Train Loss)', linewidth=2)
plt.plot(epochs, val_losses, 'r-o', label='Pérdida Validación (Val Loss)', linewidth=2)

plt.title('Evolución de la Pérdida (Loss / MSE) por Época', fontsize=14, pad=15)
plt.xlabel('Épocas', fontsize=12)
plt.ylabel('Pérdida (Cross Entropy Loss)', fontsize=12)
plt.xticks(epochs)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import torch
import os

# Intentar cargar los mejores pesos guardados durante el entrenamiento
if os.path.exists("mejor_modelo_crnn.pth"):
    print("🔄 Cargando los mejores pesos del modelo guardados ('mejor_modelo_crnn.pth')...")
    model.load_state_dict(torch.load("mejor_modelo_crnn.pth", map_location=device))
else:
    print("⚠️ No se encontró 'mejor_modelo_crnn.pth', se usará el estado actual del modelo.")

model.eval()
correct = 0
total = 0

# --- 1. EVALUACIÓN GLOBAL (Precisión en todo el set de validación) ---
print("⏳ Evaluando el set de validación (esto puede tomar un momento en CPU)...\n")
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs, _ = model(inputs)

        # Obtener la clase predicha (la de mayor probabilidad)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"🎯 Precisión Total en Validación: {accuracy:.2f}% ({correct}/{total} canciones correctas)\n")
print("="*50)
print("📊 DETALLE DE CANCIONES Y SUS GRÁFICOS DE ATENCIÓN:")
print("="*50)

# --- 2. EVALUACIÓN DETALLADA Y VISUAL (Primer batch de validación) ---
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs, attention_weights = model(inputs)

        _, predicted = torch.max(outputs, 1)

        # Iterar sobre las canciones de este batch
        for i in range(inputs.size(0)):
            pred_idx = predicted[i].item()
            true_idx = labels[i].item()

            # Traducir índices a nombres de géneros
            pred_genre = class_names[pred_idx]
            true_genre = class_names[true_idx]

            print(f"\n🎵 Canción {i+1}:")
            print(f"   • Género Real:    {true_genre}")
            print(f"   • Género Predicho: {pred_genre}")

            if pred_idx == true_idx:
                print("   • Resultado:      ¡Correcto! ✅")
            else:
                print("   • Resultado:      Incorrecto ❌")

            # attention_weights tiene forma [Batch, Reduced_Time, 1]
            weights_np = attention_weights[i].cpu().squeeze().numpy()

            # Graficar los momentos a los que la red prestó atención
            plt.figure(figsize=(10, 3))
            plt.plot(weights_np, color='tab:blue', linewidth=2)
            plt.title(f"Atención Temporal | Real: [{true_genre}] vs Predicho: [{pred_genre}]")
            plt.xlabel("Eje Temporal Reducido (Frames)")
            plt.ylabel("Peso de Importancia")
            plt.grid(True, linestyle='--', alpha=0.5)
            plt.show()

        break  # Rompemos el loop para mostrar solo las canciones del primer batch

In [ ]:
import matplotlib.pyplot as plt
import torch

model.eval()
error_count = 0
max_errors_to_show = 5

print("🔍 Buscando y analizando errores de clasificación en el conjunto de validación...\n")

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs, attention_weights = model(inputs)

        _, predicted = torch.max(outputs, 1)

        for i in range(inputs.size(0)):
            pred_idx = predicted[i].item()
            true_idx = labels[i].item()

            # Solo nos interesan las predicciones incorrectas
            if pred_idx != true_idx:
                error_count += 1
                pred_genre = class_names[pred_idx]
                true_genre = class_names[true_idx]

                print(f"❌ Caso de Error #{error_count}:")
                print(f"   • Género Real:    {true_genre.upper()}")
                print(f"   • Género Predicho: {pred_genre.upper()}")

                # Extraer pesos de atención para este audio específico
                weights_np = attention_weights[i].cpu().squeeze().numpy()

                # Graficar curva de atención temporal
                plt.figure(figsize=(10, 3.5))
                plt.plot(weights_np, color='tab:red', linewidth=2, label='Foco de Atención')
                plt.title(f"Análisis de Error: Real [{true_genre}] vs Predicho [{pred_genre}]", fontsize=11, color='darkred')
                plt.xlabel("Eje Temporal Reducido (Frames)", fontsize=9)
                plt.ylabel("Peso de Importancia", fontsize=9)
                plt.grid(True, linestyle='--', alpha=0.5)
                plt.legend()
                plt.tight_layout()
                plt.show()

                if error_count >= max_errors_to_show:
                    break
        if error_count >= max_errors_to_show:
            break

if error_count == 0:
    print("🎉 ¡Increíble! El modelo clasificó correctamente todas las canciones de este lote.")

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

model.eval()
true_labels = []
pred_labels = []

# Obtener predicciones en el set de validación
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs, _ = model(inputs)
        _, predicted = torch.max(outputs, 1)

        true_labels.extend(labels.numpy())
        pred_labels.extend(predicted.cpu().numpy())

# Calcular la matriz de confusión
cm = confusion_matrix(true_labels, pred_labels)

# Graficar
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("Matriz de Confusión - Clasificación de Géneros Musicales (CRNN)", fontsize=14, pad=15)
plt.xlabel("Género Predicho", fontsize=12, labelpad=10)
plt.ylabel("Género Real", fontsize=12, labelpad=10)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()